# Stress Level Classification (research notebook)

**Authoritative metrics** come from scripts/run_audit.py / the stress_detection package (Experiment A).  
Legacy notebook train/test demos below are **exploratory only** and are **not** benchmarks.

Historical ~94% SVM scores used **label-driven zero imputation** (target leakage) and are **invalid**.

Install: python -m pip install -e ".[dev,benchmark]" (XGBoost optional via benchmark extra).


In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier

warnings.filterwarnings('ignore', category=FutureWarning, module='seaborn')

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    XGBClassifier = None
    HAS_XGB = False
    print('XGBoost not installed; skipping XGB cells. Install with: pip install -e ".[benchmark]"')

ROOT = Path.cwd()
if (ROOT / 'src' / 'stress_detection').is_dir():
    sys.path.insert(0, str(ROOT / 'src'))
elif (ROOT.parent / 'src' / 'stress_detection').is_dir():
    sys.path.insert(0, str(ROOT.parent / 'src'))
    ROOT = ROOT.parent

from stress_detection.data import prepare_data, VALID_STRESS_LABELS
from stress_detection.models import audit_model_registry, lr_pipeline, svm_linear_pipeline, rf_pipeline
from stress_detection.evaluation import EvalConfig, run_audit_cv, per_class_metrics
from stress_detection.preprocessing import fit_imputer_median, transform_imputer

CSV = ROOT / 'StressLevelDataset_original.csv'
print('CSV:', CSV)
print('HAS_XGB:', HAS_XGB)


## Load primary analysis frame

Uses prepare_data(scope="primary"): rows 0–1099, **excludes** missing/illegal labels (no label imputation). Original CSV is never overwritten.


In [ ]:
prep = prepare_data(CSV, scope='primary')
X = prep.X.copy()
y = prep.y.copy()
print('rows', len(y), 'features', list(X.columns))
print('label counts', pd.Series(y).value_counts().sort_index().to_dict())
print('csv_sha256_raw', prep.csv_sha256_raw)
print('csv_sha256_normalized_lf', prep.csv_sha256_normalized_lf)
X.head()


## Descriptive EDA (not used for outer-fold feature selection)

Bins cover the full provisional ranges; labels are **neutral score bands**, not clinical grades. Pearson correlations are descriptive only.


In [ ]:
eda = X.copy()
eda['stress_level'] = y
anxiety_bins = [0, 5, 10, 15, 20, 22]
depression_bins = [0, 5, 10, 15, 20, 28]
band_labels = ['band0', 'band1', 'band2', 'band3', 'band4']
eda['anxiety_band'] = pd.cut(eda['anxiety_level'], bins=anxiety_bins, labels=band_labels, right=False)
eda['depression_band'] = pd.cut(eda['depression'], bins=depression_bins, labels=band_labels, right=False)
assert eda['anxiety_band'].notna().sum() + eda['anxiety_level'].isna().sum() == len(eda)
print('anxiety band counts', eda['anxiety_band'].value_counts(dropna=False).to_dict())
print('breathing_problem includes 0:', int((eda['breathing_problem'] == 0).sum()), 'missing', int(eda['breathing_problem'].isna().sum()))
# blood_pressure is an encoded field — do not treat means as mmHg
print('blood_pressure value counts', eda['blood_pressure'].value_counts().sort_index().to_dict())
corr = eda[['anxiety_level', 'depression', 'self_esteem', 'stress_level']].corr(method='pearson')
corr


## Authoritative Experiment A (package API)

Shared RepeatedStratifiedKFold; impute/scale inside pipelines on training folds only.


In [ ]:
# Smoke-sized CV inside the notebook for speed; full audit: scripts/run_audit.py
cfg = EvalConfig(n_splits=3, n_repeats=1, seed=0)
models = audit_model_registry(prep.feature_names, random_state=cfg.seed)
cv_out = run_audit_cv(prep.X, prep.y, prep.source_row_ids, models, cfg)
print('Experiment A (notebook smoke CV) — not a substitute for scripts/run_audit.py full run')
for name, stats in cv_out['summary'].items():
    print(f"{name}: macro-F1 {stats['macro_f1_mean']:.3f} ± {stats.get('macro_f1_std', 0):.3f}")


## Legacy demos (exploratory; not benchmarks)

Pipelines fit **only on the training split**. Missing labels are already excluded via prepare_data.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

def report_df(y_true, y_pred):
    rep = classification_report(
        y_true, y_pred, labels=[0, 1, 2], output_dict=True, zero_division=0
    )
    rows = {str(c): rep[str(c)] for c in [0, 1, 2] if str(c) in rep}
    return pd.DataFrame(rows).T

# Dummy baseline
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
yp = dummy.predict(X_test)
print('Dummy accuracy', accuracy_score(y_test, yp), 'macro-F1', f1_score(y_test, yp, average='macro'))

# Logistic Regression via Pipeline (no full-data scaling)
lr = lr_pipeline(random_state=0)
param_grid = {
    'logisticregression__C': [0.1, 1, 10],
    'logisticregression__penalty': ['l2'],
    'logisticregression__solver': ['lbfgs', 'saga'],
}
gs = GridSearchCV(lr, param_grid, cv=3, scoring='f1_macro')
gs.fit(X_train, y_train)
y_lr = gs.predict(X_test)
print('LR best', gs.best_params_, 'acc', accuracy_score(y_test, y_lr), 'macro-F1', f1_score(y_test, y_lr, average='macro'))
print(classification_report(y_test, y_lr, labels=[0, 1, 2], zero_division=0))
rdf = report_df(y_test, y_lr)
ax = rdf[['precision', 'recall', 'f1-score']].plot(kind='bar', figsize=(8, 4), title='LR per-class (legacy split)')
plt.tight_layout()
plt.show()


In [ ]:
# Random Forest — Pipeline; no fit_transform on full data before split
rf = rf_pipeline(random_state=0)
rf.fit(X_train, y_train)
y_rf = rf.predict(X_test)
print('RF acc', accuracy_score(y_test, y_rf), 'macro-F1', f1_score(y_test, y_rf, average='macro'))
print(classification_report(y_test, y_rf, labels=[0, 1, 2], zero_division=0))


In [ ]:
# Linear SVM (fixed C=0.2 audit config) — NO label-driven zero map
svm = svm_linear_pipeline(C=0.2)
svm.fit(X_train, y_train)
y_svm = svm.predict(X_test)
print('SVM acc', accuracy_score(y_test, y_svm), 'macro-F1', f1_score(y_test, y_svm, average='macro'))
print(classification_report(y_test, y_svm, labels=[0, 1, 2], zero_division=0))
# Audit-only leakage path remains in stress_detection.data.apply_label_driven_zero_map — not used here.


In [ ]:
# Optional XGBoost
if HAS_XGB:
    xgb = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf', XGBClassifier(
            n_estimators=100,
            max_depth=4,
            learning_rate=0.1,
            objective='multi:softprob',
            num_class=3,
            eval_metric='mlogloss',
            random_state=0,
        )),
    ])
    xgb.fit(X_train, y_train)
    y_xgb = xgb.predict(X_test)
    print('XGB acc', accuracy_score(y_test, y_xgb), 'macro-F1', f1_score(y_test, y_xgb, average='macro'))
    print(classification_report(y_test, y_xgb, labels=[0, 1, 2], zero_division=0))
else:
    print('Skipped XGBoost (optional dependency).')


## Historical note

Do **not** treat old Notebook SVM ~94% / LR ~88% figures as valid. They mixed leakage and non-nested preprocessing. Use 
esults/ from 
un_audit.py after a clean commit.
